# Human-in-the-loop
通过检查每个工具调用是否符合可配置的策略来实现这一点。如果需要干预，中间件会发出中断以暂停执行。图状态使用 LangGraph 的持久层保存 ，因此执行可以安全地暂停并在稍后恢复。

操作可以按原样批准（）、在执行前进行修改（）或拒绝并提供反馈（）。 `approve` `edit` `reject`
## Interrupt decision types  中断决策类型
`approve` `edit` `reject`
> 编辑工具参数时，请谨慎修改。对原始参数进行重大修改可能会导致模型重新评估其方法，并可能多次执行该工具或采取意外操作。
## Configuring interrupts  配置中断

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware 
from langgraph.checkpoint.memory import InMemorySaver 


agent = create_agent(
    model="gpt-4.1",
    # tools=[write_file_tool, execute_sql_tool, read_data_tool],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "write_file": True,  # All decisions (approve, edit, reject) allowed
                "execute_sql": {"allowed_decisions": ["approve", "reject"]},  # No editing allowed
                # Safe operation, no approval needed
                "read_data": False,
            },
            # Prefix for interrupt messages - combined with tool name and args to form the full message
            # e.g., "Tool execution pending approval: execute_sql with query='DELETE FROM...'"
            # Individual tools can override this by specifying a "description" in their interrupt config
            description_prefix="Tool execution pending approval",
        ),
    ],
    # Human-in-the-loop requires checkpointing to handle interrupts.
    # In production, use a persistent checkpointer like AsyncPostgresSaver.
    checkpointer=InMemorySaver(),
)

## Responding to interrupts  应对中断

In [ ]:
from langgraph.types import Command

# Human-in-the-loop leverages LangGraph's persistence layer.
# You must provide a thread ID to associate the execution with a conversation thread,
# so the conversation can be paused and resumed (as is needed for human review).
config = {"configurable": {"thread_id": "some_id"}}
# Run the graph until the interrupt is hit.
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Delete old records from the database",
            }
        ]
    },
    config=config,
    version="v2",
)

# result is a GraphOutput with .value and .interrupts
print(result.interrupts)
# > (
# >    Interrupt(
# >       value={
# >          'action_requests': [
# >             {
# >                'name': 'execute_sql',
# >                'arguments': {'query': 'DELETE FROM records WHERE created_at < NOW() - INTERVAL \'30 days\';'},
# >                'description': 'Tool execution pending approval\n\nTool: execute_sql\nArgs: {...}'
# >             }
# >          ],
# >          'review_configs': [
# >             {
# >                'action_name': 'execute_sql',
# >                'allowed_decisions': ['approve', 'reject']
# >             }
# >          ]
# >       }
# >    ),
# > )


# Resume with approval decision
agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ),
    config=config, # Same thread ID to resume the paused conversation
    version="v2",
)

# Resume with edit decision
agent.invoke(
    Command(
        # Decisions are provided as a list, one per action under review.
        # The order of decisions must match the order of actions
        # in the interrupt request.
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "new_tool_name",
                        # Arguments to pass to the tool.
                        "args": {"key1": "new_value", "key2": "original_value"},
                    }
                }
            ]
        }
    ),
    config=config,  # Same thread ID to resume the paused conversation
    version="v2",
)

# Resume with reject decision
# 这条信息会作为反馈添加到对话中，帮助客服人员理解操作被拒绝的原因以及应该采取的替代操作
agent.invoke(
    Command(
        # Decisions are provided as a list, one per action under review.
        # The order of decisions must match the order of actions
        # in the interrupt request.
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation about why the action was rejected
                    "message": "No, this is wrong because ..., instead do this ...",
                }
            ]
        }
    ),
    config=config,  # Same thread ID to resume the paused conversation
    version="v2",
)

### Multiple decisions  多项决定
当需要审查多个操作时，请按照中断中出现的顺序，对每个操作做出决定：

In [ ]:
{
    "decisions": [
        {"type": "approve"},
        {
            "type": "edit",
            "edited_action": {
                "name": "tool_name",
                "args": {"param": "new_value"}
            }
        },
        {
            "type": "reject",
            "message": "This action is not allowed"
        }
    ]
}

## Execution lifecycle  执行生命周期
中间件定义了一个钩子，该钩子在模型生成响应之后、任何工具调用执行之前运行：

1. 代理调用模型生成响应。
2. 中间件会检查响应中是否存在工具调用。
3. 如果任何调用需要人工输入，中间件会构建一个 HITLRequest，并调用 interrupt(action_requests, review_configs)
4. 智能体等待人类做出决定。
5. 根据这些决定，中间件执行已批准或已编辑的调用，合成被拒绝调用的 HITLResponse ToolMessage ，并恢复执行。

## Custom HITL logic  自定义 HITL 逻辑
对于更专业的流程，可以直接使用中断原语和中间件抽象来构建自定义 HITL 逻辑。